## Chapter 05: Embedding LLMs within Your Applications

### Getting started with LangChain

#### 1. Models and prompts

In [4]:
import environment
import importlib

importlib.reload(environment)

from langchain_deepseek import ChatDeepSeek

In [6]:
# LangChain model integrations
llm = ChatDeepSeek(
  model="deepseek-chat",
  api_key=environment.DEEPSEEK_API_KEY,
  temperature=0,
  max_tokens=None,
  timeout=None,
  max_retries=2,
)

In [7]:
print(
  llm.invoke("Tell me a joke").content
)

Why did the scarecrow win an award?  
Because he was outstanding in his field.


In [8]:
# Prompt templates
from langchain_core.prompts import PromptTemplate

In [9]:
# Instantiation
template = """
Sentence: {sentence}

Translation in {language}:
"""

prompt = PromptTemplate.from_template(template)

print(
  prompt.format(
    sentence = "The cat is on the table.",
    language = "spanish"
	)
)


Sentence: The cat is on the table.

Translation in spanish:



In [10]:
# Example selector
from langchain_core.example_selectors import BaseExampleSelector

In [11]:
example = BaseExampleSelector.add_example(
  self=BaseExampleSelector,
  example={
    "prompt": "What is your name?",
    "completion": "I go by Sally.",
	}
)

In [12]:
print(example)

None


#### 2. Data connections

- Document loaders

In [14]:
from langchain_community.document_loaders import CSVLoader

In [15]:
loader = CSVLoader(file_path="./data/sample.csv")
data = loader.load()

In [18]:
print(data[0].page_content)

Name: John
Age: 25
City: New York


- Document transformers

In [19]:
with open("./data/mountain.txt") as f:
  mountain = f.read()

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [21]:
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=100, 				# number of characters for each chunk
  chunk_overlap=20, 			# number of characters overlapping between a preceding and following chunk
  length_function=len, 		# function used to measure the number of characters
)

texts = text_splitter.create_documents([mountain])

In [22]:
print(texts[0].page_content)
print(texts[1].page_content)
print(texts[2].page_content)

Amidst the serene landscape, towering mountains stand as majestic guardians of nature's beauty.
The crisp mountain air carries whispers of tranquility, while the rustling leaves compose a
leaves compose a symphony of wilderness.


- Text embedding models

In [23]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [24]:
embedding_model = GoogleGenerativeAIEmbeddings(
  model="models/gemini-embedding-001",
  api_key=environment.GEMINI_API_KEY
)

# Batch (embed multiple strings at once for a processing speedup)
embeddings = embedding_model.embed_documents(
  [
    "Good morning!",
    "Oh, hello!",
    "I want to report an accident",
    "Sorry to hear that. May I ask your name?",
    "Sure. Mario Rossi."
	]
)

In [25]:
print("Embed documents:")
print(
  f"Number of vectors: {len(embeddings)}; Dimension of each vector: {len(embeddings[0])}"
)
print("-" * 50)

embedded_query = embedding_model.embed_query(
  "What was the name mentioned in the conversation?"
)

print("Embed query:")
print(
  f"Dimension of the vector: {len(embedded_query)}"
)
print(
  f"Sample of the first 5 elements of the vector: {embedded_query[:5]}"
)

Embed documents:
Number of vectors: 5; Dimension of each vector: 3072
--------------------------------------------------
Embed query:
Dimension of the vector: 3072
Sample of the first 5 elements of the vector: [-0.030619625, 0.0032838325, 0.015690394, -0.0870063, -0.008714505]


- Vector stores (Vector DBs)

In [27]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

In [28]:
# Load the document, split it into chunks, embed each chunk and load it into the vector store
raw_documents = TextLoader("./data/dialogue.txt").load()
text_splitter = RecursiveCharacterTextSplitter(
  chunk_size=50,
  chunk_overlap=0,
  # separator="\n",
  add_start_index=True
)
documents = text_splitter.split_documents(raw_documents)
embedding_model = GoogleGenerativeAIEmbeddings(
  model="models/gemini-embedding-001",
  api_key=environment.GEMINI_API_KEY
)
vector_store = InMemoryVectorStore(embedding=embedding_model)

vector_store.add_documents(documents=documents)

['c6917667-0cd2-4d09-b3e6-14242a933050',
 '05374417-a8bd-4dff-b708-3efe3d04543c',
 'cec3e14d-bfef-47df-bc18-a9ce28af39ac',
 '68dd53eb-94c2-4412-8d3e-afc878c0105b']

In [30]:
query = "What is the reason for calling?"
search_query = vector_store.similarity_search(query)

print(search_query[0].page_content)

Sorry to hear that. May I ask your name?


- Retrievers

In [36]:
retriever = vector_store.as_retriever()
query = "What was the reason of the call?"

retriever.invoke(query)

[Document(id='cec3e14d-bfef-47df-bc18-a9ce28af39ac', metadata={'source': './data/dialogue.txt', 'start_index': 54}, page_content='Sorry to hear that. May I ask your name?'),
 Document(id='05374417-a8bd-4dff-b708-3efe3d04543c', metadata={'source': './data/dialogue.txt', 'start_index': 25}, page_content='I want to report an accident'),
 Document(id='c6917667-0cd2-4d09-b3e6-14242a933050', metadata={'source': './data/dialogue.txt', 'start_index': 0}, page_content='Good morning!\nOh, hello!'),
 Document(id='68dd53eb-94c2-4412-8d3e-afc878c0105b', metadata={'source': './data/dialogue.txt', 'start_index': 95}, page_content='Sure, Mario Rossi.')]